In [1]:
from dateutil.utils import today

from tensym.interpreter.lexer._lexer import tokenize_string
import os

os.environ['TENSYM_LEXER_DEBUG_MODE'] = '1'
os.environ['TENSYM_PARSER_DEBUG_MODE'] = '1'

from tensym.kernal import TenKernal

symkernal = TenKernal()



In [2]:
test_e2e = """

chart Spherical := [t, r, theta, phi]
metric g on Spherical

g_{mu}_{nu} := diag(-A(r), B(r), r**2, r**2*sin(theta)**2)

def Coordinates: {
    sp := [t, r, theta, phi]
}

def Constants: {
    G,
    c
}

def Metric g: {

}


let f(x) := x**2 + 2*x + 1 # Example function definition

print(f(3) + f(4)) # Example function call

dsolve( f(x) = 0, x ) # Example equation solving

let Eq0 := f(x) = 0



let eq1_solution := dsolve( Eq0, x ) # Example equation solving with a named equation

def g_{mu}_{nu} := [
                 [-A(r),0,0,0],
                 [0,B(r),0,0],
                 [0,0,r**2,0],
                 [0,0,0,r**2*sin(theta)**2]
               ]

# Now we have defined the metric above, we can call any individual component of the Ricci tensor itself (as it is metric dependent)

let eq0 := Ric_{mu:0}_{nu:0}
let eq1 := Ric_{mu:1}_{nu:1}
let eq2 := Ric_{mu:2}_{nu:2}
let Eq1 := eq0 = eq1
let eq5 := (eq0*B(r) + eq1*A(r))*(r*B(r))


let B := RHS( dsolve(eq5, B(r)) )

let eq6 = simplify( subs(eq2, B(r), B) )

let A := RHS( dsolve(eq6, A(r)) )

let g_{mu}_{nu} := [
             [A,0,0,0],
             [0,1/A,0,0],
             [0,0,r**2,0],
             [0,0,0,r**2*sin(theta)**2]
           ]

# Step 5: We prove that C_1 and C_2 equations are in terms of c, G, M by comparing with Newton at large radius

let a := C^{a:1}_{b:0 c:0}

solve( a*c**2 + G*M/r**2 ) # This shows us what

let array_1 := [
                  [-(1 - (2 * G * M) / (c**2*r)), 0, 0, 0],
                  [0, 1 / (1 - (2 * G * M) / (c**2*r)), 0, 0],
                  [0, 0, r**2, 0],
                  [0, 0, 0, r**2 * sin(theta) ** 2]
              ]

background Schild:
    coords [t, r, theta, phi]
    g_{mu}_{nu} := array_1


fn f(x):
    x**2 + 2*x + 1

let Gamma^{a}_{c f} := (1/2) * g^{a b} * ( d_{c} * g_{b f} + d_{f} * g_{b c} - d_{b} * g_{c f} )
let Riemann^{a}_{m b n} := d_{b}*Gamma^{a}_{n m} + Gamma^{a}_{b l}*Gamma^{l}_{n m} - d_{n}*Gamma^{a}_{b m} - Gamma^{a}_{n l}*Gamma^{l}_{b m}
let Ricci_{m n} := Riemann^{a}_{m a n}
let T1^{a f h i} := g^{i d}*(g^{h c}*(g^{f b}*Riemann^{a}_{b c d}))
let T2_{a f h i} := g_{a n}*Riemann^{n}_{f h i}
let S := T1^{a f h i} * T2_{a f h i}


with metric g as Minkowski(M=1, c=1):
    tsimplify( S )

with metric g as Schwarzschild:
    tsimplify( S )

with Schwarzschild {
    tsimplify( S )
}


"""

test_one = """
A_{a} := x + y
"""
tuple([ord(_) for _ in "not"])

(110, 111, 116)

In [15]:
token = tokens.advance()

In [19]:
token.lexeme

':'

In [4]:
# tensym_poc.py
from __future__ import annotations
from dataclasses import dataclass, asdict
from typing import Any, Dict, List, Tuple, Optional, Protocol, Literal
import json
import numpy as np
import sympy as sp
from concurrent.futures import ProcessPoolExecutor, as_completed
import itertools as it
import os

# =========================
# Operation dataclasses
# =========================

@dataclass(frozen=True)
class Operation:
    """Base class for all bytecode operations."""

    @property
    def opname(self) -> str:
        return type(self).__name__

    def to_record(self) -> Tuple[str, Dict[str, Any]]:
        return self.opname, asdict(self)

    @staticmethod
    def from_record(name: str, payload: Dict[str, Any]) -> "Operation":
        cls = _OP_REGISTRY[name]
        return cls(**payload)

# --- Core ops ---

@dataclass(frozen=True)
class TEN_ALLOC(Operation):
    buffer_id: str
    shape: Tuple[int, ...]
    kind: Literal["object", "numeric"] = "object"
    # variance: +1 = contravariant (upper), -1 = covariant (lower)
    variance: Tuple[int, ...] = ()

@dataclass(frozen=True)
class TEN_FILL_CONST(Operation):
    buffer_id: str
    loc: Tuple[int, ...]
    const_val: Any

@dataclass(frozen=True)
class TEN_FILL_SYM(Operation):
    buffer_id: str
    loc: Tuple[int, ...]
    sym_val: str  # store as string; reconstruct Symbol/Expr later if you like

@dataclass(frozen=True)
class TEN_MAKE_VIEW(Operation):
    view_id: str
    base_id: str
    perm: Tuple[int, ...]  # permutation of axes

@dataclass(frozen=True)
class TEN_EVAL_KERNEL(Operation):
    kernel_id: str  # look up a ComponentKernelPlan from constant_pool

@dataclass(frozen=True)
class TEN_BARRIER(Operation):
    pass

@dataclass(frozen=True)
class TEN_STORE_GLOBAL(Operation):
    name: str
    buffer_id: str

@dataclass(frozen=True)
class TEN_RETURN(Operation):
    buffer_id: Optional[str] = None

_OP_REGISTRY = {
    cls.__name__: cls
    for cls in [TEN_ALLOC, TEN_FILL_CONST, TEN_FILL_SYM, TEN_MAKE_VIEW,
                TEN_EVAL_KERNEL, TEN_BARRIER, TEN_STORE_GLOBAL, TEN_RETURN]
}

# =========================
# Bytecode program
# =========================

@dataclass
class BytecodeProgram:
    """Holds a list of operations + a (Python-side) constant pool."""
    ops: List[Operation]
    # Python-only pool (kernels, literals, symbols, metrics, etc.)
    const_pool: Dict[str, Any]

    # --- Serialization of ops to a newline-delimited text file ---
    def to_text(self) -> str:
        """Serialize ops to a simple line format: OPNAME|<json-payload> per line."""
        lines = []
        for op in self.ops:
            name, payload = op.to_record()
            lines.append(name + "|" + json.dumps(payload, separators=(",", ":")))
        return "\n".join(lines)

    @staticmethod
    def from_text(text: str, const_pool: Optional[Dict[str, Any]] = None) -> "BytecodeProgram":
        ops: List[Operation] = []
        for line in text.strip().splitlines():
            if not line.strip():
                continue
            try:
                name, payload_json = line.split("|", 1)
            except ValueError:
                raise ValueError(f"Malformed line (missing '|'): {line}")
            payload = json.loads(payload_json)
            ops.append(Operation.from_record(name, payload))
        return BytecodeProgram(ops=ops, const_pool=const_pool or {})

    def save(self, path: str) -> None:
        with open(path, "w", encoding="utf-8") as f:
            f.write(self.to_text())

    @staticmethod
    def load(path: str, const_pool: Optional[Dict[str, Any]] = None) -> "BytecodeProgram":
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        return BytecodeProgram.from_text(text, const_pool=const_pool)

# =========================
# Kernel plan & backend API
# =========================

@dataclass(frozen=True)
class ComponentKernelPlan:
    """A self-contained plan to compute a tensor result via contraction/sum-of-products."""
    kernel_id: str
    out_buffer: str
    input_buffers: Tuple[str, ...]  # e.g. ("G", "T")
    # labels
    free_labels: Tuple[str, ...]        # e.g. ("a", "c", "d")
    contracted_labels: Tuple[str, ...]  # e.g. ("b",)
    # mappings from labels to axis indices
    out_axis_of_label: Dict[str, int]         # {"a":0,"c":1,"d":2}
    in_axis_of_label: Tuple[Dict[str, int], ...]  # len == len(input_buffers)
    # sizes for each label (assume rectangular domains)
    axis_sizes: Dict[str, int]           # {"a":4,"b":4,"c":4,"d":4}
    # execution hints
    tile: Optional[Tuple[int, ...]] = None
    simplify_each: bool = True

    # Precomputed strides for fast ravel (optional convenience)
    out_shape: Tuple[int, ...] = ()
    out_strides: Tuple[int, ...] = ()

    def ravel(self, idx: Tuple[int, ...]) -> int:
        """Map an n-D index to a flat offset for the output buffer."""
        return sum(i * s for i, s in zip(idx, self.out_strides))

class ExecContext:
    def __init__(
            self,
            parallel: Literal["auto", "processes", "threads"]="auto",
            max_workers: Optional[int]=None,
            tile: Optional[Tuple[int, ...]]=None
    ):
        self.parallel = parallel
        self.max_workers = max_workers or (os.cpu_count() or 4)
        self.tile = tile

class Backend(Protocol):
    def alloc(self, shape: Tuple[int, ...], kind: Literal["object", "numeric"]) -> Any: ...
    def read(self, buf: Any, idx: Tuple[int, ...]) -> Any: ...
    def write(self, buf: Any, idx: Tuple[int, ...], value: Any) -> None: ...
    def zero(self) -> Any: ...
    def add(self, a: Any, b: Any) -> Any: ...
    def mul(self, a: Any, b: Any) -> Any: ...
    def simplify(self, x: Any) -> Any: ...

class SymPyBackend:
    __slots__ = ()
    def alloc(self, shape: Tuple[int, ...], kind: Literal["object", "numeric"]="object") -> Any:
        if kind != "object":
            raise NotImplementedError("This PoC backend is symbolic-only (object dtype).")
        a = np.empty(shape, dtype=object)
        # Initialize with zeros to keep r/w simple; you may prefer None
        z = sp.Integer(0)
        it = np.nditer(a, flags=['multi_index', 'refs_ok'], op_flags=['writeonly'])
        for x in it:
            x[...] = z
        return a
    def read(self, buf: Any, idx: Tuple[int, ...]) -> Any: return buf[idx]
    def write(self, buf: Any, idx: Tuple[int, ...], value: Any) -> None: buf[idx] = value
    def zero(self) -> Any: return sp.Integer(0)
    def add(self, a: Any, b: Any) -> Any: return a + b
    def mul(self, a: Any, b: Any) -> Any: return a * b
    def simplify(self, x: Any) -> Any: return sp.simplify(x)

# =========================
# Process-pool kernel executor (tiles, exclusive writes)
# =========================

# Globals inside worker processes (set via initializer)
_W_BACKEND: Optional[Backend] = None
_W_INPUTS: Optional[Tuple[Any, ...]] = None
_W_PLAN: Optional[ComponentKernelPlan] = None

def _worker_init(backend: Backend, inputs: Tuple[Any, ...], plan: ComponentKernelPlan):
    global _W_BACKEND, _W_INPUTS, _W_PLAN
    _W_BACKEND = backend
    _W_INPUTS = inputs
    _W_PLAN = plan

def _tiles(shape: Tuple[int, ...], tile: Tuple[int, ...]):
    ranges = [range(0, dim, t) for dim, t in zip(shape, tile)]
    for starts in it.product(*ranges):
        ends = tuple(min(s + t, dim) for s, t, dim in zip(starts, tile, shape))
        yield starts, ends

def _compute_tile(task: Tuple[Tuple[int, ...], Tuple[int, ...]]):
    """Runs inside a worker. Returns (starts, ends, dense_tile_ndarray)."""
    (starts, ends) = task
    b = _W_BACKEND
    plan = _W_PLAN
    inputs = _W_INPUTS
    assert b and plan and inputs

    # Output axes for the free labels, in output-buffer axis order
    out_idx_axes = [plan.out_axis_of_label[lbl] for lbl in plan.free_labels]
    tile_shape = tuple(ends[ax] - starts[ax] for ax in out_idx_axes)
    tile_buf = np.empty(tile_shape, dtype=object)

    # Ranges over free + contracted labels
    free_ranges = [range(starts[ax], ends[ax]) for ax in out_idx_axes]
    red_ranges  = [range(plan.axis_sizes[lbl]) for lbl in plan.contracted_labels]

    in_maps = plan.in_axis_of_label  # per-operand label->axis dict

    for free_vals in it.product(*free_ranges):
        # Bind free labels once for this output component
        free_bindings = {lbl: free_vals[i] for i, lbl in enumerate(plan.free_labels)}
        # Compute local tile index (in tile_buf coordinates)
        local_idx = tuple(free_vals[i] - starts[out_idx_axes[i]] for i in range(len(out_idx_axes)))

        acc = b.zero()
        for red_vals in it.product(*red_ranges):
            red_bindings = {lbl: red_vals[i] for i, lbl in enumerate(plan.contracted_labels)}
            term = None

            # Multiply all operands at this (free, red) point
            for op_i, in_map in enumerate(in_maps):
                # Build the operand's own index tuple in axis order
                rank = (max(in_map.values()) + 1) if in_map else 0
                idx = [0] * rank
                for lbl, axis in in_map.items():
                    if lbl in free_bindings:
                        idx[axis] = free_bindings[lbl]
                    elif lbl in red_bindings:
                        idx[axis] = red_bindings[lbl]
                    else:
                        # Label neither free nor contracted -> should not happen
                        raise RuntimeError(f"Unbound label '{lbl}' for operand {op_i}")
                comp = b.read(inputs[op_i], tuple(idx))
                term = comp if term is None else b.mul(term, comp)

            acc = b.add(acc, term) if term is not None else acc

        if plan.simplify_each:
            acc = b.simplify(acc)

        tile_buf[local_idx] = acc

    return starts, ends, tile_buf

def eval_kernel_sympy(plan: ComponentKernelPlan,
                      buffers: Dict[str, Any],
                      backend: Backend,
                      ctx: ExecContext):
    out = buffers[plan.out_buffer]
    inputs = tuple(buffers[i] for i in plan.input_buffers)

    # Choose a reasonable tile (defaults to ~2^k volume)
    tile = ctx.tile or plan.tile or tuple(max(1, min(32, s)) for s in plan.out_shape)
    tasks = list(_tiles(plan.out_shape, tile))

    max_workers = ctx.max_workers
    with ProcessPoolExecutor(
        max_workers=max_workers,
        initializer=_worker_init,
        initargs=(backend, inputs, plan),
    ) as pool:
        futs = [pool.submit(_compute_tile, t) for t in tasks]
        for fut in as_completed(futs):
            starts, ends, tile_buf = fut.result()
            # Place dense tile into output buffer (exclusive region, no locks)
            # Build slices in output axis order
            sl = [slice(None)] * out.ndim
            for lbl in plan.free_labels:
                ax = plan.out_axis_of_label[lbl]
                sl[ax] = slice(starts[ax], ends[ax])
            out[tuple(sl)] = tile_buf

# =========================
# The VM
# =========================

class VM:
    def __init__(self, backend: Backend, ctx: Optional[ExecContext] = None):
        self.backend = backend
        self.ctx = ctx or ExecContext()
        self.buffers: Dict[str, Any] = {}
        self.globals: Dict[str, Any] = {}

    def run(self, program: BytecodeProgram) -> Any:
        cp = program.const_pool
        result = None
        for op in program.ops:
            if isinstance(op, TEN_ALLOC):
                self.buffers[op.buffer_id] = self.backend.alloc(op.shape, op.kind)
            elif isinstance(op, TEN_FILL_CONST):
                self.backend.write(self.buffers[op.buffer_id], op.loc, op.const_val)
            elif isinstance(op, TEN_FILL_SYM):
                # Simple reconstruction: Symbols by name, or parse as sympy Expr if you wish.
                val = sp.sympify(op.sym_val)
                self.backend.write(self.buffers[op.buffer_id], op.loc, val)
            elif isinstance(op, TEN_MAKE_VIEW):
                # PoC: demonstrate axis permutation via numpy view
                base = self.buffers[op.base_id]
                self.buffers[op.view_id] = np.transpose(base, axes=op.perm)
            elif isinstance(op, TEN_EVAL_KERNEL):
                plan: ComponentKernelPlan = cp[op.kernel_id]
                eval_kernel_sympy(plan, self.buffers, self.backend, self.ctx)
            elif isinstance(op, TEN_STORE_GLOBAL):
                self.globals[op.name] = self.buffers[op.buffer_id]
            elif isinstance(op, TEN_BARRIER):
                pass  # sequencing point; all ops are synchronous in this PoC
            elif isinstance(op, TEN_RETURN):
                result = self.buffers.get(op.buffer_id) if op.buffer_id else None
                break
            else:
                raise NotImplementedError(op.opname)
        return result

# =========================
# Build the requested program:
#   G_ab (4x4, lower-lower), T^b_cd (4x4x4, upper-lower-lower)
#   R_acd = sum_b G_ab * T^b_cd  (result 4x4x4 lower-lower-lower)
# =========================

def build_program_G_times_T_contract_b() -> BytecodeProgram:
    ops: List[Operation] = []

    # 1) Allocate G_ab
    ops.append(TEN_ALLOC(buffer_id="G", shape=(4, 4), kind="object",
                         variance=(-1, -1)))  # lower, lower

    # 2) Allocate T^b_cd
    ops.append(TEN_ALLOC(buffer_id="T", shape=(4, 4, 4), kind="object",
                         variance=(+1, -1, -1)))  # upper, lower, lower

    # 3) Allocate R_acd (result)
    ops.append(TEN_ALLOC(buffer_id="R", shape=(4, 4, 4), kind="object",
                         variance=(-1, -1, -1)))  # lower, lower, lower

    # 4) Kernel plan: R_{a c d} = sum_b G_{a b} * T^{b}{}_{c d}
    # Axis label sizes
    axis_sizes = {"a": 4, "b": 4, "c": 4, "d": 4}

    # Label -> axis maps for operands and output
    G_map = {"a": 0, "b": 1}          # G_ab
    T_map = {"b": 0, "c": 1, "d": 2}  # T^b_cd
    R_map = {"a": 0, "c": 1, "d": 2}  # R_acd

    out_shape = (axis_sizes["a"], axis_sizes["c"], axis_sizes["d"])
    # Row-major simple stride calc
    strides = (out_shape[1] * out_shape[2], out_shape[2], 1)

    plan = ComponentKernelPlan(
        kernel_id="K_GxT_contract_b",
        out_buffer="R",
        input_buffers=("G", "T"),
        free_labels=("a", "c", "d"),
        contracted_labels=("b",),
        out_axis_of_label=R_map,
        in_axis_of_label=(G_map, T_map),
        axis_sizes=axis_sizes,
        tile=(2, 2, 2),         # small demo tile; tune as needed
        simplify_each=True,
        out_shape=out_shape,
        out_strides=strides,
    )

    const_pool: Dict[str, Any] = {"K_GxT_contract_b": plan}

    # 5) Evaluate kernel
    ops.append(TEN_EVAL_KERNEL(kernel_id="K_GxT_contract_b"))

    # 6) Store global & return
    ops.append(TEN_STORE_GLOBAL(name="Result", buffer_id="R"))
    ops.append(TEN_RETURN(buffer_id="R"))

    return BytecodeProgram(ops=ops, const_pool=const_pool)

# =========================
# Example: build & serialize
# =========================


prog = build_program_G_times_T_contract_b()
text = prog.to_text()
print("=== Bytecode Text ===")
print(text)
# Recover from text (const_pool must be wired separately)
prog2 = BytecodeProgram.from_text(text, const_pool=prog.const_pool)

# Execute (with empty buffers, result will be zeros unless you fill G/T)
vm = VM(backend=SymPyBackend(), ctx=ExecContext(parallel="processes", max_workers=os.cpu_count(), tile=(2,2,2)))
result = vm.run(prog2)
print("Result shape:", np.shape(result))


=== Bytecode Text ===
TEN_ALLOC|{"buffer_id":"G","shape":[4,4],"kind":"object","variance":[-1,-1]}
TEN_ALLOC|{"buffer_id":"T","shape":[4,4,4],"kind":"object","variance":[1,-1,-1]}
TEN_ALLOC|{"buffer_id":"R","shape":[4,4,4],"kind":"object","variance":[-1,-1,-1]}
TEN_EVAL_KERNEL|{"kernel_id":"K_GxT_contract_b"}
TEN_STORE_GLOBAL|{"name":"Result","buffer_id":"R"}
TEN_RETURN|{"buffer_id":"R"}
Result shape: (4, 4, 4)


In [8]:
import sympy as sp

# Symbols; assume Gii are negative real numbers for convergence
x, y, z = sp.symbols('x y z', real=True)
G11 : sp.Symbol = sp.symbols('G11', real=True, negative=True)
G22 = sp.symbols('G22', real=True, negative=True)
G33 = sp.symbols('G33', real=True, negative=True)


In [2]:

# Integrand
prefactor = 1/sp.sqrt(-(2*sp.pi)**3 * G11*G22*G33)
exponent  = sp.exp((x**2)/(2*G11) + (y**2)/(2*G22) + (z**2)/(2*G33))
integrand = prefactor * exponent

# Perform the triple integral over the entire real line
result = sp.integrate(
    sp.integrate(
        sp.integrate(
            integrand,
            (x, -sp.oo, sp.oo)
        ),
        (y, -sp.oo, sp.oo)
    ),
    (z, -sp.oo, sp.oo)
)
print(result)  # returns 1


1


In [9]:
from relative

vm.execute(
    """
    declare x, y, z :: Real
    declare G_11, G_22, G_33 :: Real
    declare G_11 < 0
    declare G_22 < 0
    declare G_33 < 0

    let f(x, y, z) := 1/sqrt(-(2*pi)**3 * G_11*G_22*G_33) * exp( (x**2)/(2*G_11) + (y**2)/(2*G_22) + (z**2)/(2*G_33) )
    int_{-oo}^{oo} f(x, y, z) dx dy dz
    """
)

NameError: name 'VM' is not defined